# Track1 End-to-End Learner Notebook

이 노트북은 **Track1 실습(Mission 1~5)** 전체를 한 번에 따라갈 수 있도록 구성된 참가자용 워크벤치입니다.

- Mission 1: 비즈니스 질문 정리
- Mission 2: 데이터 프로파일링
- Mission 3: 표준 스키마/코드 정규화
- Mission 4: Ontology 엔터티/관계 설계
- Mission 5: 매핑/검증 + Track2 인계 패키지 초안


## Track1 기술 맥락과 참조 문서

Track1의 기술 목표는 정형 원천 데이터를 **의미 모델(Ontology)** 로 승격해 FabricIQ 질의의 안정성을 확보하는 것입니다.

- 통합 설계 기준: [Fabric_Ontology_AI_Workshop_Integrated_Plan_v2.0.md](../common/docs/Fabric_Ontology_AI_Workshop_Integrated_Plan_v2.0.md)
- 트랙 실습 기준(DoD/미션): [WORKBOOK.md](./WORKBOOK.md)
- 데이터 구조/관계 상세: [Track1_Data_Structure_Detailed_Guide.md](./docs/Track1_Data_Structure_Detailed_Guide.md)
- 온톨로지 모델링 개념: [Track1_Ontology_Concepts_and_Graph_Design_Guide.md](./docs/Track1_Ontology_Concepts_and_Graph_Design_Guide.md)
- 데이터셋 계약/품질 규칙: [track1/data/README.md](./data/README.md)

핵심 기술 축:
1. **질문 중심 모델링**: Q1~Q5를 엔터티/관계 경로로 고정
2. **품질 내재화**: 프로파일링 결과를 표준화 규칙으로 연결
3. **그래프화**: 물리 FK + 논리(다중 홉) 관계를 함께 설계
4. **검증 가능성**: SQL 기준값으로 온톨로지 경로의 재현성 확보


In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
import csv
import json
import sqlite3

def find_track1_data_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "customers.csv").exists() and (base / "orders.csv").exists():
            return base
        candidate = base / "track1" / "data"
        if (candidate / "customers.csv").exists() and (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError("track1/data 루트를 찾을 수 없습니다.")

DATA_ROOT = find_track1_data_root()
WORKBENCH_DIR = DATA_ROOT / "generated" / "workbench"
WORKBENCH_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"WORKBENCH_DIR: {WORKBENCH_DIR}")


## Mission 1. 비즈니스 질문 정리 (Q1~Q5)

Track1 공통 질문을 확정하고 질문별 핵심 테이블을 연결합니다.

기술 포인트:
- 질문을 KPI와 데이터 경로로 분해해야 이후 Ontology 관계가 흔들리지 않습니다.
- 질문-테이블 매핑은 Track3 에이전트의 Tool 라우팅 기준이 됩니다.

관련 문서:
- [WORKBOOK.md](./WORKBOOK.md)
- [Track1_Data_Structure_Detailed_Guide.md#advanced-scenario](./docs/Track1_Data_Structure_Detailed_Guide.md#advanced-scenario)
- [Track1_Data_Structure_Detailed_Guide.md#source-table-structure](./docs/Track1_Data_Structure_Detailed_Guide.md#source-table-structure)


In [ ]:
question_map = [
    {
        "id": "Q1",
        "question": "결제 실패가 캠페인 전환율에 미치는 영향은 무엇인가?",
        "tables": ["campaigns", "campaign_attribution", "customers", "orders", "payments"],
    },
    {
        "id": "Q2",
        "question": "배송 지연은 반품률과 고객 만족도에 어떤 영향을 미치는가?",
        "tables": ["shipments", "returns", "support_tickets", "orders", "customers", "channels"],
    },
    {
        "id": "Q3",
        "question": "프로모션 유형별 할인 전략이 매출총이익과 재구매율에 미치는 영향은 무엇인가?",
        "tables": ["promotions", "order_promotions", "orders", "order_items", "products", "customers"],
    },
    {
        "id": "Q4",
        "question": "재고 부족/품절 경험은 주문 취소율과 고객센터 문의량에 어떤 영향을 미치는가?",
        "tables": ["inventory_snapshots", "products", "orders", "support_tickets", "channels"],
    },
    {
        "id": "Q5",
        "question": "채널·고객등급별 반품 사유 패턴은 재구매율에 어떤 차이를 만드는가?",
        "tables": ["returns", "orders", "customers", "channels", "order_items", "products"],
    },
]

for row in question_map:
    print(f"[{row['id']}] {row['question']}")
    print("  tables:", ", ".join(row["tables"]))

(WORKBENCH_DIR / "mission1_question_map.json").write_text(
    json.dumps(question_map, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission1_question_map.json")


## Mission 2. 데이터 프로파일링 (결측/중복/이상값)

CSV 14개를 SQLite 메모리 DB로 로드해 실습지의 프로파일링 쿼리를 실행합니다.

기술 포인트:
- Track1 데이터는 의도적으로 노이즈가 삽입되어 있으며, 품질 결함을 탐지해 표준화 규칙의 근거로 사용합니다.
- 결측/중복/이상값/코드 분포를 별도로 수집해야 규칙 영향 범위를 분리할 수 있습니다.

관련 문서:
- [track1/data/README.md](./data/README.md)
- [Track1_Data_Structure_Detailed_Guide.md#profiling-checkpoints](./docs/Track1_Data_Structure_Detailed_Guide.md#profiling-checkpoints)
- [Track1_Instructor_Data_Answer_Key.md](./docs/Track1_Instructor_Data_Answer_Key.md)


In [ ]:
TABLE_FILES = {
    "customers": "customers.csv",
    "products": "products.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "returns": "returns.csv",
    "channels": "channels.csv",
    "payments": "payments.csv",
    "shipments": "shipments.csv",
    "inventory_snapshots": "inventory_snapshots.csv",
    "promotions": "promotions.csv",
    "order_promotions": "order_promotions.csv",
    "campaigns": "campaigns.csv",
    "campaign_attribution": "campaign_attribution.csv",
    "support_tickets": "support_tickets.csv",
}

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

def quote_ident(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'

row_count = {}
for table, filename in TABLE_FILES.items():
    path = DATA_ROOT / filename
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        cols = reader.fieldnames or []
        col_defs = ", ".join(f"{quote_ident(c)} TEXT" for c in cols)
        conn.execute(f"CREATE TABLE {quote_ident(table)} ({col_defs});")
        placeholders = ", ".join(["?"] * len(cols))
        insert_sql = f"INSERT INTO {quote_ident(table)} ({', '.join(quote_ident(c) for c in cols)}) VALUES ({placeholders});"
        rows = [tuple((r.get(c) if r.get(c) != "" else None) for c in cols) for r in reader]
        conn.executemany(insert_sql, rows)
        row_count[table] = len(rows)

print("Loaded tables:")
for t in sorted(row_count):
    print(f"- {t}: {row_count[t]}")


In [ ]:
def run_query(sql: str) -> list[dict[str, object]]:
    cur = conn.execute(sql)
    return [dict(r) for r in cur.fetchall()]

profiling_queries = {
    "payments_null_rate": """
        SELECT
          SUM(CASE WHEN payment_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_rate_payment_id,
          SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_rate_order_id,
          SUM(CASE WHEN payment_status IS NULL OR payment_status = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_rate_payment_status
        FROM payments;
    """,
    "shipment_pk_duplicates": """
        SELECT shipment_id, COUNT(*) AS cnt
        FROM shipments
        GROUP BY shipment_id
        HAVING COUNT(*) > 1
        ORDER BY cnt DESC;
    """,
    "anomaly_counts": """
        SELECT 'products.unit_price<=0' AS issue, COUNT(*) AS c FROM products WHERE CAST(unit_price AS REAL) <= 0
        UNION ALL SELECT 'promotions.discount<0', COUNT(*) FROM promotions WHERE CAST(discount_amount AS REAL) < 0
        UNION ALL SELECT 'inventory.on_hand<0', COUNT(*) FROM inventory_snapshots WHERE CAST(on_hand_qty AS REAL) < 0
        UNION ALL SELECT 'order_items.qty<=0', COUNT(*) FROM order_items WHERE CAST(quantity AS REAL) <= 0;
    """,
}

profiling_results = {}
for name, sql in profiling_queries.items():
    profiling_results[name] = run_query(sql)
    print(f"\n[{name}]")
    for row in profiling_results[name]:
        print(row)

(WORKBENCH_DIR / "mission2_profiling_results.json").write_text(
    json.dumps(profiling_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission2_profiling_results.json")


## Mission 3. 표준 스키마/코드 정규화 설계

비표준 상태값을 표준 코드셋으로 변환하는 매핑 초안을 만듭니다.

기술 포인트:
- 표준화는 단순 치환이 아니라, 분석 단위(상태/재시도/시점)를 분리해 의미를 보존하는 과정입니다.
- 키 규칙(`*_id`), 타입 규칙(Date/DateTime), 코드 규칙을 동시에 맞춰야 Ontology 속성 충돌이 줄어듭니다.

관련 문서:
- [Track1_Data_Structure_Detailed_Guide.md#standardization-rules](./docs/Track1_Data_Structure_Detailed_Guide.md#standardization-rules)
- [WORKBOOK.md](./WORKBOOK.md)


### Mission 3 체크포인트

- `orders.order_status`, `payments.payment_status`, `shipments.shipment_status`의 원본 분포를 먼저 확인합니다.
- 표준 코드셋으로 매핑할 때 **의미가 섞인 값**(예: `RetrySuccess`)은 상태/재시도 여부를 분리해 기록합니다.
- Track2에서 같은 키워드/상태를 사용하므로, 표준화 규칙은 인계 패키지에 반드시 포함합니다.

참조:
- [Track1_Data_Structure_Detailed_Guide.md#mapping-3step](./docs/Track1_Data_Structure_Detailed_Guide.md#mapping-3step)
- [PREREQUISITES.md](../track2/PREREQUISITES.md)


In [ ]:
ORDER_STATUS_MAP = {
    "Completed": "PAID",
    "Cancelled": "CANCELLED",
    "NEW": "NEW",
    "PAID": "PAID",
    "SHIPPED": "SHIPPED",
    "RETURNED": "RETURNED",
}

PAYMENT_STATUS_MAP = {
    "Success": "AUTHORIZED",
    "RetrySuccess": "AUTHORIZED",
    "Failed": "FAILED",
    "INITIATED": "INITIATED",
    "AUTHORIZED": "AUTHORIZED",
    "FAILED": "FAILED",
    "REFUNDED": "REFUNDED",
}

SHIPMENT_STATUS_MAP = {
    "Delivered": "DELIVERED",
    "Delayed": "DELAYED",
    "InTransit": "IN_TRANSIT",
    "READY": "READY",
    "DELIVERED": "DELIVERED",
    "DELAYED": "DELAYED",
}

def status_distribution(table: str, col: str) -> list[tuple[str, int]]:
    rows = run_query(
        f"SELECT COALESCE({col}, '<NULL>') AS v, COUNT(*) AS c FROM {table} GROUP BY COALESCE({col}, '<NULL>') ORDER BY c DESC"
    )
    return [(str(r['v']), int(r['c'])) for r in rows]

for table, col, mapping in [
    ("orders", "order_status", ORDER_STATUS_MAP),
    ("payments", "payment_status", PAYMENT_STATUS_MAP),
    ("shipments", "shipment_status", SHIPMENT_STATUS_MAP),
]:
    print(f"\n[{table}.{col}] raw distribution")
    dist = status_distribution(table, col)
    for raw, cnt in dist:
        print(f"- {raw}: {cnt} -> {mapping.get(raw, '<UNMAPPED>')}")

standardization_plan = {
    "key_rule": "<entity>_id (snake_case)",
    "datetime_rule": "DATE/TIMESTAMP type normalization",
    "order_status_map": ORDER_STATUS_MAP,
    "payment_status_map": PAYMENT_STATUS_MAP,
    "shipment_status_map": SHIPMENT_STATUS_MAP,
}
(WORKBENCH_DIR / "mission3_standardization_plan.json").write_text(
    json.dumps(standardization_plan, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission3_standardization_plan.json")


## Mission 4. Ontology 엔터티/관계 설계

권장 기준(엔터티 14개, 관계 20개 + 확장 논리관계)을 구조화합니다.

기술 포인트:
- 물리 테이블 관계(FK)와 의미 관계(논리 경로)를 분리해 모델링해야 설명 가능성이 높아집니다.
- Track3 Tool 결합 질의를 고려해 경로 기반(예: Campaign->Order->Payment->Shipment->Return)으로 설계합니다.

관련 문서:
- [Track1_Data_Structure_Detailed_Guide.md#ontology-model](./docs/Track1_Data_Structure_Detailed_Guide.md#ontology-model)
- [Track1_Ontology_Concepts_and_Graph_Design_Guide.md](./docs/Track1_Ontology_Concepts_and_Graph_Design_Guide.md)
- [Track1_Instructor_Script_v1.0.md](./docs/Track1_Instructor_Script_v1.0.md)


### Mission 4 설계 원칙

- 엔터티는 명사형, 관계는 동사형으로 작성합니다.
- 물리 FK 관계와 논리(다중 홉) 관계를 구분해 기록합니다.
- 강사 대본 연계 확장 관계도 함께 포함합니다.
  - `Payment relates_to Return (logical)`
  - `Shipment relates_to Return (logical)`
  - `Order applies Promotion (logical path)`
- 핵심 경로 예시: `Campaign -> Order -> Payment -> Shipment -> Return`


In [ ]:
entities = [
    "Customer", "Product", "Channel", "Campaign", "Promotion", "Order", "OrderItem",
    "Payment", "Shipment", "Return", "SupportTicket", "InventorySnapshot",
    "OrderPromotion", "CampaignAttribution",
]

relationships_core = [
    "Customer places Order",
    "Order belongs_to Channel",
    "Order has Payment",
    "Order fulfilled_by Shipment",
    "Order includes OrderItem",
    "OrderItem references Product",
    "Order has Return",
    "Return references Product",
    "Return requested_by Customer",
    "Customer raises SupportTicket",
    "SupportTicket relates_to Order",
    "Product has InventorySnapshot",
    "Order receives OrderPromotion",
    "OrderPromotion points_to Promotion",
    "Campaign drives CampaignAttribution",
    "CampaignAttribution points_to Order",
    "CampaignAttribution points_to Customer",
    "Promotion influences Order (logical)",
    "Campaign influences Order (logical)",
    "Customer purchases Product (logical)",
]

relationships_extension = [
    "Payment relates_to Return (logical)",
    "Shipment relates_to Return (logical)",
    "Order applies Promotion (logical path)",
]

print("entity_count:", len(entities))
print("core_relationship_count:", len(relationships_core))
print("extension_relationship_count:", len(relationships_extension))
print("\nDoD range check (10-16 entities, 15-25 relationships):")
print("- entities_ok:", 10 <= len(entities) <= 16)
print("- relationships_ok:", 15 <= len(relationships_core) <= 25)

(WORKBENCH_DIR / "mission4_entities.json").write_text(
    json.dumps(entities, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(WORKBENCH_DIR / "mission4_relationships_core.json").write_text(
    json.dumps(relationships_core, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(WORKBENCH_DIR / "mission4_relationships_extension.json").write_text(
    json.dumps(relationships_extension, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved mission4 artifacts under", WORKBENCH_DIR)


In [ ]:
mapping_template_csv = WORKBENCH_DIR / "mission5_mapping_template.csv"
with mapping_template_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["entity", "source_table", "source_column", "standard_column", "ontology_property", "note"])
    writer.writerow(["Order", "orders", "order_status", "order_status_std", "Order.status", "status code normalization"])
    writer.writerow(["Payment", "payments", "payment_status", "payment_status_std", "Payment.status", "RetrySuccess -> AUTHORIZED"])
    writer.writerow(["Shipment", "shipments", "shipment_status", "shipment_status_std", "Shipment.status", "CamelCase -> UPPER_SNAKE"])
    writer.writerow(["Campaign", "campaign_attribution", "campaign_id", "campaign_id", "Campaign.campaign_id", "bridge mapping"])
    writer.writerow(["SupportTicket", "support_tickets", "ticket_reason", "ticket_reason_std", "SupportTicket.reason", "controlled vocabulary"])

print("Saved mapping template:", mapping_template_csv)


## Mission 5. 검증 쿼리 실행 + Track2 인계 패키지

실습지 기대 검출값(참조 2건, PK중복 1건, 금액불일치 2건)을 확인합니다.

기술 포인트:
- 검증은 모델의 신뢰구간을 정량화하는 단계이며, Track2 품질 게이트의 입력 근거가 됩니다.
- 참조무결성/중복/값정합성은 각각 관계 오류, 식별자 오류, 집계 오류를 대표합니다.

관련 문서:
- [Track1_Data_Structure_Detailed_Guide.md#validation-structure](./docs/Track1_Data_Structure_Detailed_Guide.md#validation-structure)
- [WORKBOOK.md](./WORKBOOK.md)
- [PREREQUISITES.md](../track2/PREREQUISITES.md)


In [ ]:
validation_queries = {
    "fk_payments_order": """
        SELECT p.payment_id
        FROM payments p
        LEFT JOIN orders o ON p.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """,
    "fk_support_customer": """
        SELECT t.ticket_id
        FROM support_tickets t
        LEFT JOIN customers c ON t.customer_id = c.customer_id
        WHERE c.customer_id IS NULL;
    """,
    "pk_duplicate_shipment": """
        SELECT shipment_id, COUNT(*) AS cnt
        FROM shipments
        GROUP BY shipment_id
        HAVING COUNT(*) > 1;
    """,
    "amount_mismatch_order_items": """
        SELECT o.order_id, o.gross_amount, SUM(CAST(i.sales_amount AS REAL)) AS items_sum
        FROM orders o
        JOIN order_items i ON o.order_id = i.order_id
        GROUP BY o.order_id, o.gross_amount
        HAVING ABS(CAST(o.gross_amount AS REAL) - SUM(CAST(i.sales_amount AS REAL))) > 0.01;
    """,
}

validation_results = {k: run_query(v) for k, v in validation_queries.items()}
summary = {
    "fk_errors_total": len(validation_results["fk_payments_order"]) + len(validation_results["fk_support_customer"]),
    "pk_duplicate_total": len(validation_results["pk_duplicate_shipment"]),
    "amount_mismatch_total": len(validation_results["amount_mismatch_order_items"]),
}

print("Validation summary:")
print(json.dumps(summary, ensure_ascii=False, indent=2))

expected = {
    "fk_errors_total": 2,
    "pk_duplicate_total": 1,
    "amount_mismatch_total": 2,
}
print("\nExpectation match:")
for k, v in expected.items():
    print(f"- {k}: actual={summary[k]} expected={v} ok={summary[k] == v}")

(WORKBENCH_DIR / "mission5_validation_results.json").write_text(
    json.dumps({"summary": summary, "details": validation_results}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("\nSaved:", WORKBENCH_DIR / "mission5_validation_results.json")


In [ ]:
team = "Team-A"
workspace_id = "<WORKSPACE_ID>"
ontology_id = "<ONTOLOGY_ID>"

track2_handoff = f"""[TRACK2_HANDOFF_PACKAGE]
team={team}
handoffAtKst=<YYYY-MM-DD HH:MM>
workspaceId={workspace_id}
ontologyId={ontology_id}
ontologyName=retail_track1_ontology_v1
entityCount={len(entities)}
relationshipCount={len(relationships_core)}
corePaths=Campaign->Order->Payment;Order->Shipment->Return;Promotion->Order->Margin
mappingHighlights=Order:orders.order_status;Payment:payments.payment_status;Shipment:shipments.shipment_status;Campaign:campaign_attribution.campaign_id;SupportTicket:support_tickets.ticket_reason
openIssues=FK orphan 2건|정합성 리스크|Track2에서 링크/키워드 검증 강화;Shipment PK 중복 1건|집계 중복 위험|중복제거 규칙 적용;Gross mismatch 2건|지표 왜곡 가능|검증 리포트 경고 포함
workiqKeys=캠페인명,SummerPush,VIPRetention,FlashWeek,BackToSchool;상품명,AeroPhone X,SmartWatch Pro,UltraBook 15,DailyTee Cotton;고객등급,Platinum
evidenceLinks=generated/workbench/mission5_validation_results.json
[/TRACK2_HANDOFF_PACKAGE]"""

handoff_path = WORKBENCH_DIR / "TRACK2_HANDOFF_PACKAGE.txt"
handoff_path.write_text(track2_handoff, encoding="utf-8")
print(track2_handoff)
print("\nSaved:", handoff_path)


## 기술 연계 관점의 다음 단계

Track1 산출물은 아래 기술 계약으로 Track2/Track3에 전달됩니다.

1. Ontology 식별/모델 요약/핵심 경로를 인계해 WorkIQ 검색 키를 고정
2. 검증 결과를 품질 리스크로 전달해 Track2 점수화 기준에 반영
3. 논리 관계 경로를 Track3 Tool 결합 질의 설계의 입력으로 사용

관련 문서:
- [WORKBOOK.md](./WORKBOOK.md)
- [PREREQUISITES.md](../track2/PREREQUISITES.md)
- [WORKBOOK.md](../track2/WORKBOOK.md)
- [PREREQUISITES.md](../track3/PREREQUISITES.md)
- [WORKBOOK.md](../track3/WORKBOOK.md)
